In [2]:
import pandas as pd

In [9]:
df_original = pd.read_csv('Children Recode_final.csv')
df_original['Malnurished'] = df_original[['Underweight', 'Stunting', 'Wasting']].max(axis =1)
df_original.drop(['Underweight', 'Stunting', 'Wasting'], axis = 1, inplace=True)
df_original.head()

,Child_age,Mother_education,Age_first_sex,Pregnancy_terminated,Wealth_index,Place_residence,BMI,Children_under5,Total_children_ever_born,Child_sex,...,Ethnicity_2,Ethnicity_3,Ethnicity_4,Ethnicity_5,Ethnicity_6,Ethnicity_7,Ethnicity_8,Ethnicity_9,Ethnicity_10,Malnurished
0,17,1,14,0,1,2,22.00,1,1,2,...,1,0,0,0,0,0,0,0,0,0
1,40,2,17,1,2,2,25.10,2,2,1,...,0,0,0,0,0,0,1,0,0,1
2,59,2,17,1,2,2,25.10,2,2,2,...,0,0,0,0,0,0,1,0,0,1
3,55,2,17,0,2,2,21.53,1,1,2,...,0,0,0,0,0,0,1,0,0,1
4,14,1,16,0,1,2,28.03,1,1,1,...,0,0,0,0,0,0,1,0,0,0


In [13]:
from sklearn.model_selection import train_test_split

X = df_original.drop(columns=['Malnurished'])
y = df_original['Malnurished']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [38]:
from sklearn.preprocessing import MinMaxScaler
columns_to_scale = ['Child_age', 'Age_first_sex', 'BMI', 'Mother_age_current', 'Mother_age_at_first_birth']
scaler = MinMaxScaler()

# X_train_scaled = X_train.copy()
# X_test_scaled = X_test.copy()

X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

In [39]:
X_train.head()

,Child_age,Mother_education,Age_first_sex,Pregnancy_terminated,Wealth_index,Place_residence,BMI,Children_under5,Total_children_ever_born,Child_sex,...,Religion_5,Ethnicity_2,Ethnicity_3,Ethnicity_4,Ethnicity_5,Ethnicity_6,Ethnicity_7,Ethnicity_8,Ethnicity_9,Ethnicity_10
1582,0.915254,2,0.185185,1,1,2,0.210629,2,2,2,...,0,0,0,0,1,0,0,0,0,0
2165,0.966102,1,0.259259,0,5,1,0.414615,2,2,1,...,0,0,0,1,0,0,0,0,0,0
1387,0.084746,2,0.444444,1,5,1,0.451348,0,2,1,...,0,0,0,1,0,0,0,0,0,0
178,1.000000,2,0.333333,0,3,2,0.653380,1,2,2,...,0,0,0,0,1,0,0,0,0,0
478,0.864407,0,0.148148,1,4,1,0.472841,1,1,2,...,0,0,0,0,0,0,0,0,0,1


In [40]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [41]:
y_pred = model.predict(X_test)
pd.crosstab(y_test, y_pred)

col_0,0,1
Malnurished,,
0,292,14
1,122,20


In [42]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.71      0.95      0.81       306
           1       0.59      0.14      0.23       142

    accuracy                           0.70       448
   macro avg       0.65      0.55      0.52       448
weighted avg       0.67      0.70      0.63       448



In [45]:
from sklearn.model_selection import GridSearchCV, cross_val_score
# Define the hyperparameters and their values
param_grid = {
    'penalty': ['l1', 'l2'],
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'saga'],
    'max_iter': [100, 200, 300]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, n_jobs=-1, scoring='accuracy')

# Fit GridSearchCV
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_

# Evaluate the best model
best_model = grid_search.best_estimator_
score = best_model.score(X_test, y_test)

# Cross-validation
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='accuracy')

print("Best parameters found: ", best_params)
print("Cross-validation scores: ", cv_scores)
print("Model accuracy: ", score)

Best parameters found:  {'C': 0.1, 'max_iter': 100, 'penalty': 'l1', 'solver': 'liblinear'}
Cross-validation scores:  [0.65625    0.52008929 0.66071429 0.55133929 0.68232662]
Model accuracy:  0.6808035714285714
